# 🏠 Property Finder Egypt Scraper
### Sale & Rent Listings

### Made By: Eng. Elsayed Bakry
This notebook scrapes property listings from **Property Finder Egypt** (`propertyfinder.eg`).

**How to use:**
1. Run **Cell 1** once to install dependencies.
2. Run **Cell 2** to import libraries.
3. Run **Cell 3** to define the shared `get_driver()` helper.
4. Run **Cell 4 / 5** for Sale / Rent listings — each is fully independent.
5. Run **Cell 6** to merge all scraped data into one CSV.

> ⚙️ **Settings** — change `MAX_PAGES` (default **50**) and `OUTPUT_FILE` at the top of each scraper cell.
> 🔇 To run without opening a browser window, uncomment the `--headless=new` line inside `get_driver()`.


## Cell 1 — Install dependencies
Run this **once**. Skip on subsequent runs if packages are already installed.

In [1]:
!pip install -q selenium pandas webdriver-manager pytz



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Users\5g\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Cell 2 — Imports
Standard libraries used by every scraper below.

In [2]:
import re
import time
import pandas as pd
from datetime import datetime, timedelta
import pytz

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

print("✅ Imports ready")


✅ Imports ready


## Cell 3 — Shared helper
One reusable function used by every scraper:

- **`get_driver()`** — launches a Chrome browser with anti-bot settings.


In [3]:
# ─────────────────────────────────────────────────────────────────
# get_driver() — returns a configured Chrome WebDriver instance
# ─────────────────────────────────────────────────────────────────

def get_driver():
    """Launch Chrome with settings that reduce bot-detection."""
    options = Options()

    # Uncomment the next line to run without opening a browser window:
    # options.add_argument("--headless=new")

    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    # Mask navigator.webdriver flag
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"}
    )
    return driver


print("✅ Helper functions defined")

✅ Helper functions defined


## Cell 4a — Scrape Agencies
Scrapes **41 pages** of agencies from `propertyfinder.eg/ar/find-broker/search`.
Stores agency **logo URL** and **name** in `agencies_df`.
This DataFrame is later used to match agency logos from property listings to real names.


In [11]:
# ── Settings ──────────────────────────────────────────────────────
AGENCY_BASE_URL  = "https://www.propertyfinder.eg/ar/find-broker/search"
AGENCY_MAX_PAGES = 41
# ──────────────────────────────────────────────────────────────────


def normalize_logo_url(url):
    """Remove resolution segment (e.g. /260x200, /178x98) from logo URLs
    so the same logo at different sizes maps to the same lookup key."""
    if not url:
        return None
    import re
    return re.sub(r'/\d+x\d+(\.\w+)$', r'\1', url)


def get_logo_url(driver, img_el):
    """Extract real logo URL, handling lazy-loaded placeholder gifs."""
    PLACEHOLDER = ("data:image", "data:application")

    def is_real(u):
        return bool(u) and not any(u.startswith(p) for p in PLACEHOLDER)

    # 1. src
    src = img_el.get_attribute("src") or ""
    if is_real(src):
        return src

    # 2. data-src
    data_src = img_el.get_attribute("data-src") or ""
    if is_real(data_src):
        return data_src

    # 3. srcset — take first URL
    srcset = img_el.get_attribute("srcset") or ""
    if srcset:
        first = srcset.split(",")[0].strip().split(" ")[0]
        if is_real(first):
            return first

    # 4. Scroll into view and retry once
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", img_el)
    time.sleep(0.4)
    src = img_el.get_attribute("src") or ""
    if is_real(src):
        return src

    return None


def scrape_agencies(base_url=AGENCY_BASE_URL, max_pages=AGENCY_MAX_PAGES):
    driver       = get_driver()
    all_agencies = []
    seen_names   = set()

    for page in range(1, max_pages + 1):
        page_url = f"{base_url}?page={page}"
        print(f"\n{'='*40}")
        print(f"  AGENCIES PAGE {page}  →  {page_url}")
        print(f"{'='*40}")

        driver.get(page_url)

        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'li[data-testid="list-card"]'))
            )
        except Exception:
            print("⚠️  Timed-out waiting for agency cards — stopping.")
            break

        # Scroll slowly down the page to trigger lazy-loading for all images
        page_height = driver.execute_script("return document.body.scrollHeight")
        for y in range(0, page_height, 400):
            driver.execute_script(f"window.scrollTo(0, {y});")
            time.sleep(0.08)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.5)

        cards = driver.find_elements(By.CSS_SELECTOR, 'li[data-testid="list-card"]')
        print(f"Found {len(cards)} agency cards")

        if not cards:
            print("⚠️  No cards found — stopping.")
            break

        for card in cards:
            try:
                try:
                    name = card.find_element(By.TAG_NAME, "h2").text.strip()
                except Exception:
                    name = None

                try:
                    img_el   = card.find_element(By.CSS_SELECTOR, 'img[data-testid="card-image"]')
                    raw_logo = get_logo_url(driver, img_el)
                    logo_url = normalize_logo_url(raw_logo)
                except Exception:
                    logo_url = None

                dedup_key = name or logo_url
                if dedup_key and dedup_key in seen_names:
                    continue
                if dedup_key:
                    seen_names.add(dedup_key)

                all_agencies.append({"agency_name": name, "agency_logo": logo_url})
                print(f"  + {name}  {'✅' if logo_url else '⚠️  no logo'}")

            except Exception as e:
                print(f"  ⚠️  Agency card error: {e}")

    driver.quit()

    agencies_df = pd.DataFrame(all_agencies)
    real_logos  = agencies_df["agency_logo"].notna().sum()
    print(f"\n✅ Agencies scraping done — {len(agencies_df)} agencies ({real_logos} with logo)")
    return agencies_df


# ── Run ───────────────────────────────────────────────────────────
agencies_df = scrape_agencies()
agencies_df.head(10)


  AGENCIES PAGE 1  →  https://www.propertyfinder.eg/ar/find-broker/search?page=1
Found 20 agency cards
  + Coldwell Banker Wealth  ✅
  + Selection Code  ✅
  + Square real estate  ✅
  + New Avenue Real Estate  ✅
  + Master Mind Real Estate Services  ✅
  + Spade consultancy  ✅
  + Egypt Best Properties  ✅
  + IRTKAZ  ✅
  + Select for real estate  ✅
  + Property Hills Star  ✅
  + Remax Professional.  ✅
  + Summit Real Estate  ✅
  + Insider Real Estate Consultancy  ✅
  + Property Hills.  ✅
  + Closer  ✅
  + Falcon for Real Estate  ✅
  + Platinum Properties  ✅
  + Allocate Real Estate  ✅
  + CBE New Homes  ✅
  + Go Green Egypt Real Estate  ✅

  AGENCIES PAGE 2  →  https://www.propertyfinder.eg/ar/find-broker/search?page=2
Found 20 agency cards
  + Abrag 2  ✅
  + Abrag Real Estate  ✅
  + Adviser Real Estate  ✅
  + Edge Real Estates  ✅
  + Citizen Home  ✅
  + AK Circle  ✅
  + Premier Property  ✅
  + Property Hills VIP  ✅
  + why solutions  ✅
  + Home Realtors  ✅
  + Next Door Consultancy  ✅


,agency_name,agency_logo
0,Coldwell Banker Wealth,https://static.shared.propertyfinder.eg/media/...
1,Selection Code,https://static.shared.propertyfinder.eg/media/...
2,Square real estate,https://static.shared.propertyfinder.eg/media/...
3,New Avenue Real Estate,https://static.shared.propertyfinder.eg/media/...
4,Master Mind Real Estate Services,https://static.shared.propertyfinder.eg/media/...
5,Spade consultancy,https://static.shared.propertyfinder.eg/media/...
6,Egypt Best Properties,https://static.shared.propertyfinder.eg/media/...
7,IRTKAZ,https://static.shared.propertyfinder.eg/media/...
8,Select for real estate,https://static.shared.propertyfinder.eg/media/...
9,Property Hills Star,https://static.shared.propertyfinder.eg/media/...


In [12]:
agencies_df.tail(10)

,agency_name,agency_logo
799,الحسين للاستثمار العقاري,https://static.shared.propertyfinder.eg/media/...
800,One Way For Real Estate,https://static.shared.propertyfinder.eg/media/...
801,البوابة المصرية الأولى,https://static.shared.propertyfinder.eg/media/...
802,INVESTORIA,https://static.shared.propertyfinder.eg/media/...
803,Real Solutions,https://static.shared.propertyfinder.eg/media/...
804,المدينة المنورة للتسويق العقاري,https://static.shared.propertyfinder.eg/media/...
805,الصفا والمروة للتسويق العقاري,https://static.shared.propertyfinder.eg/media/...
806,White House,https://static.shared.propertyfinder.eg/media/...
807,El Swisy Real Estate,https://static.shared.propertyfinder.eg/media/...
808,M W Real Estate,https://static.shared.propertyfinder.eg/media/...


In [14]:
#save to CSV
agencies_df.to_csv("propertyfinder_agencies.csv", index=False)

## Cell 4 — Scrape Sale Listings
Scrapes pages **1 through MAX_PAGES** (now **30**) from Property Finder Egypt's Sale search results.
- `agency_logo` column stores the **broker logo URL**.
- `agency_name` is matched from the agencies DataFrame.
- `publish_date` / `publish_time` are computed from Arabic relative-time strings (Cairo TZ).
- All gallery images are captured (not just 3).
A seen-URLs guard stops if the site repeats the same listings.


In [5]:
# ── Settings ──────────────────────────────────────────────────────
PROPERTYFINDER_SALE_URL = "https://www.propertyfinder.eg/ar/search?c=1&fu=0&ob=mr"
MAX_PAGES               = 50         
# ──────────────────────────────────────────────────────────────────


CAIRO_TZ = pytz.timezone("Africa/Cairo")


def parse_arabic_relative_date(text):
    """
    Convert an Arabic relative-date string to a Cairo-localised datetime.

    Recognised patterns (examples):
        نُشِر منذ 30 دقيقة   → subtract 30 minutes
        نُشِر منذ 2 ساعات    → subtract 2 hours
        نُشِر منذ 5 أيام     → subtract 5 days
        نُشِر منذ أسبوع      → subtract 1 week
        نُشِر منذ 3 أسابيع   → subtract 3 weeks
        نُشِر منذ شهر        → subtract 1 month  (~30 days)
        نُشِر منذ 2 شهور     → subtract 2 months (~60 days)
        نُشِر منذ سنة        → subtract 1 year   (~365 days)
        نُشِر منذ 2 سنوات    → subtract 2 years  (~730 days)

    Returns (publish_date, publish_time) strings, e.g. ("11.06.2026", "10:00 AM").
    Returns (None, None) if parsing fails.
    """
    if not text:
        return None, None

    now_cairo = datetime.now(CAIRO_TZ)

    UNIT_MAP = [
        (r"دقيق",                  "minutes"),
        (r"ساع",                   "hours"),
        (r"يوم|أيام|يومان",        "days"),
        (r"أسبوع|أسابيع",          "weeks"),
        (r"شهر|شهور|أشهر",         "months"),
        (r"سنة|سنوات|عام|أعوام",   "years"),
    ]

    number_match = re.search(r"(\d+)", text)
    number = int(number_match.group(1)) if number_match else 1

    delta = None
    for pattern, unit_key in UNIT_MAP:
        if re.search(pattern, text):
            if unit_key == "months":
                delta = timedelta(days=30 * number)
            elif unit_key == "years":
                delta = timedelta(days=365 * number)
            else:
                delta = timedelta(**{unit_key: number})
            break

    if delta is None:
        return None, None

    pub_dt    = now_cairo - delta
    pub_date  = pub_dt.strftime("%d.%m.%Y")
    pub_time  = pub_dt.strftime("%I:%M %p")   # e.g. "10:00 AM"
    return pub_date, pub_time


def scrape_propertyfinder(base_url, listing_type, max_pages=MAX_PAGES):
    """
    Scrape property listings from Property Finder Egypt using URL-based pagination.

    Property Finder appends &page=N to the base search URL.
    Page 1  → https://…/search?...&page=1
    Page N  → https://…/search?...&page=N

    Parameters
    ----------
    base_url     : str  – Property Finder search URL (no page param)
    listing_type : str  – "Sale" or "Rent" (stored in the listing_type column)
    max_pages    : int  – Maximum number of pages (None = scrape until end)

    Returns
    -------
    pd.DataFrame with columns:
        title, price, location, property_type, bedrooms, bathrooms, area,
        listing_type, agency_logo, agency_name, publish_date, publish_time,
        listing_url, image_urls, source
    """
    driver       = get_driver()
    all_listings = []
    seen_urls    = set()

    # Build a logo→name lookup dict from agencies_df (if available)
    logo_to_name = {}
    try:
        if not agencies_df.empty:
            logo_to_name = dict(zip(agencies_df["agency_logo"], agencies_df["agency_name"]))
    except NameError:
        pass  # agencies_df not yet scraped — agency_name will be None

    for page in range(1, (max_pages or 9999) + 1):
        page_url = f"{base_url}&page={page}"

        print(f"\n{'='*40}")
        print(f"  {listing_type.upper()} PAGE {page}  →  {page_url}")
        print(f"{'='*40}")

        driver.get(page_url)

        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'article[data-testid="property-card"]'))
            )
        except Exception:
            print("⚠️  Timed-out waiting for cards — stopping.")
            break

        time.sleep(2)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        cards = driver.find_elements(By.CSS_SELECTOR, 'article[data-testid="property-card"]')
        print(f"Found {len(cards)} cards")

        if not cards:
            print("⚠️  No cards found — stopping.")
            break

        new_on_page = 0
        for idx, card in enumerate(cards):
            try:
                try:
                    listing_url = card.find_element(By.CSS_SELECTOR, 'a[data-testid="property-card-link"]').get_attribute("href")
                except Exception:
                    listing_url = None

                if listing_url and listing_url in seen_urls:
                    continue
                if listing_url:
                    seen_urls.add(listing_url)
                    new_on_page += 1

                try:
                    title = card.find_element(By.TAG_NAME, "h3").text.strip()
                except Exception:
                    title = None

                try:
                    price = card.find_element(By.CSS_SELECTOR, '[data-testid="property-card-price"]').text.strip()
                except Exception:
                    price = None

                try:
                    location = card.find_element(By.CSS_SELECTOR, '[data-testid="property-card-location"] p').text.strip()
                except Exception:
                    location = None

                try:
                    property_type = card.find_element(By.CSS_SELECTOR, '[data-testid="property-card-type"]').text.strip()
                except Exception:
                    property_type = None

                try:
                    bedrooms = card.find_element(By.CSS_SELECTOR, '[data-testid="property-card-spec-bedroom"]').text.strip()
                except Exception:
                    bedrooms = None

                try:
                    bathrooms = card.find_element(By.CSS_SELECTOR, '[data-testid="property-card-spec-bathroom"]').text.strip()
                except Exception:
                    bathrooms = None

                try:
                    area = card.find_element(By.CSS_SELECTOR, '[data-testid="property-card-spec-area"]').text.strip()
                except Exception:
                    area = None

                # ── agency_logo: broker logo image URL ─────────────────────────
                try:
                    agency_logo = card.find_element(
                        By.CSS_SELECTOR,
                        'div[data-testid="property-card-broker-logo"] img[data-testid="gallery-picture"]'
                    ).get_attribute("src")
                except Exception:
                    agency_logo = None

                # ── agency_name: matched from agencies_df logo lookup ───────────
                agency_name = logo_to_name.get(agency_logo) if agency_logo else None

                # ── publish_date / publish_time from Arabic relative text ────────
                try:
                    raw_publish = card.find_element(By.CSS_SELECTOR, "footer p").text.strip()
                except Exception:
                    raw_publish = None

                publish_date, publish_time = parse_arabic_relative_date(raw_publish)

                # ── image_urls: ALL gallery images (not just 3) ─────────────────
                try:
                    gallery_container = card.find_element(By.CSS_SELECTOR, '[data-testid="gallery"]')
                    driver.execute_script(
                        "arguments[0].scrollIntoView({block: 'center'});",
                        gallery_container
                    )
                    time.sleep(0.3)

                    gallery_imgs = card.find_elements(
                        By.CSS_SELECTOR,
                        '[data-testid="gallery"] img[data-testid="gallery-picture"]'
                    )
                    image_urls_list = []
                    for img in gallery_imgs:
                        src = img.get_attribute("src") or img.get_attribute("data-src")
                        if src and "placeholder" not in src and src not in image_urls_list:
                            image_urls_list.append(src)

                    image_urls = "|".join(image_urls_list) if image_urls_list else None
                except Exception:
                    image_urls = None

                all_listings.append({
                    "title":          title,
                    "price":          price,
                    "location":       location,
                    "property_type":  property_type,
                    "bedrooms":       bedrooms,
                    "bathrooms":      bathrooms,
                    "area":           area,
                    "listing_type":   listing_type,
                    "agency_logo":    agency_logo,
                    "agency_name":    agency_name,
                    "publish_date":   publish_date,
                    "publish_time":   publish_time,
                    "listing_url":    listing_url,
                    "image_urls":     image_urls,
                    "source":         "Property Finder Egypt",
                })

                short_title = (title[:65] + "...") if title and len(title) > 65 else (title or "Unknown")
                print(f"  {idx+1:>3}. {short_title}")

            except Exception as e:
                print(f"  ⚠️  Card {idx+1} error: {e}")

        if new_on_page == 0:
            print("\n🏁 No new listings on this page — scraping complete.")
            break

    driver.quit()

    df = pd.DataFrame(all_listings)
    print(f"\n✅ {listing_type} scraping done — {len(df)} listings collected")
    return df


# ── Run ───────────────────────────────────────────────────────────
sale_df = scrape_propertyfinder(PROPERTYFINDER_SALE_URL, "Sale")
sale_df.head()



  SALE PAGE 1  →  https://www.propertyfinder.eg/ar/search?c=1&fu=0&ob=mr&page=1
Found 25 cards
    1. فيلا TYPE D ب اسكارليت استلام فوري باقساط بحري
    2. امتلك تاون هاوس ع10سنين تلال إيست متشطب بالتكيفات
    3. تاون هاوس استلام فوري موقع مميز ف سوان ليك ريزيدنس
    4. ستند الون فيلا استلام فوري موقع مميز في سوان ليك
    5. فيلا بموقع مميز للبيع في سوان ليك ريزيدنس
    6. تاون هاوس بسعر مميز متشطب بالكامل ماونتن فيو
    7. شاليه مميز للبيع باقساط متشطب بالكامل مزارين
    8. شاليه باقساط للبيع جون الساحل الشمالي
    9. فيلا مميزة تطل علي البحر للبيع هاسيندا باي الساحل
   10. شقة ارضي153م+82م جاردن متشطبة بالتكييفات فقلب زايد
   11. شاليه للبيع هاسيندا باي متشطب بالكامل باقساط
   12. شاليه مميز جاهز أرضي بجاردن سوان ليك Best Deal
   13. فيلا للبيع فوكا باي الساحل الشمالي مفروشة بالكامل
   14. فيلا مميزة جاهزة واجهة بحري سوان ليك Best Deal
   15. توين هاوس مميز بحري فرصة نادرة سوان ليك Best Price
   16. شاليه مميز مع رووف كبير بحري علي اللاجون BEST DEAL
   17. فيلا مستقلة فاخرة للبيع با

,title,price,location,property_type,bedrooms,bathrooms,area,listing_type,agency_logo,agency_name,publish_date,publish_time,listing_url,image_urls,source
0,فيلا TYPE D ب اسكارليت استلام فوري باقساط بحري,٥٣٬٠٠٠٬٠٠٠ جنيه,"سوان ليك ريزيدنس, كمبوندات التجمع الخامس, التج...",فيلا,7+,7+,٦٥٠ متر مربع,Sale,https://static.shared.propertyfinder.eg/media/...,None,10.06.2026,06:48 AM,https://www.propertyfinder.eg/ar/plp/buy/villa...,https://static.shared.propertyfinder.eg/media/...,Property Finder Egypt
1,امتلك تاون هاوس ع10سنين تلال إيست متشطب بالتكيفات,١٥٬٤٩٠٬٠٠٠ جنيه,"تلال إيست, كمبوندات التجمع الخامس, التجمع الخا...",فيلا,3,3,٢٦٢ متر مربع,Sale,https://static.shared.propertyfinder.eg/media/...,None,11.06.2026,03:48 AM,https://www.propertyfinder.eg/ar/plp/buy/villa...,https://static.shared.propertyfinder.eg/media/...,Property Finder Egypt
2,تاون هاوس استلام فوري موقع مميز ف سوان ليك ريز...,٢٦٬٠٠٠٬٠٠٠ جنيه,"سوان ليك ريزيدنس, كمبوندات التجمع الخامس, التج...",تاون هاوس,5,4,٢٤٧ متر مربع,Sale,https://static.shared.propertyfinder.eg/media/...,None,10.06.2026,02:48 PM,https://www.propertyfinder.eg/ar/plp/buy/townh...,https://static.shared.propertyfinder.eg/media/...,Property Finder Egypt
3,ستند الون فيلا استلام فوري موقع مميز في سوان ليك,٣٩٬٠٠٠٬٠٠٠ جنيه,"سوان ليك ريزيدنس, كمبوندات التجمع الخامس, التج...",فيلا,4,5,٣٨٨ متر مربع,Sale,https://static.shared.propertyfinder.eg/media/...,None,10.06.2026,02:48 PM,https://www.propertyfinder.eg/ar/plp/buy/villa...,https://static.shared.propertyfinder.eg/media/...,Property Finder Egypt
4,فيلا بموقع مميز للبيع في سوان ليك ريزيدنس,٤٢٬٠٠٠٬٠٠٠ جنيه,"سوان ليك ريزيدنس, كمبوندات التجمع الخامس, التج...",فيلا,5,4,٤٨٥ متر مربع,Sale,https://static.shared.propertyfinder.eg/media/...,None,10.06.2026,01:48 PM,https://www.propertyfinder.eg/ar/plp/buy/villa...,https://static.shared.propertyfinder.eg/media/...,Property Finder Egypt


## Cell 5 — Scrape Rent Listings
Scrapes pages **1 through MAX_PAGES** (now **30**) from Property Finder Egypt's Rent search results
using the same `scrape_propertyfinder()` function defined above.


In [6]:
# ── Settings ──────────────────────────────────────────────────────
PROPERTYFINDER_RENT_URL = "https://www.propertyfinder.eg/ar/search?c=2&fu=0&rp=m&ob=mr"
MAX_PAGES               = 50
# ──────────────────────────────────────────────────────────────────


# ── Run ───────────────────────────────────────────────────────────
rent_df = scrape_propertyfinder(PROPERTYFINDER_RENT_URL, "Rent")
rent_df.head()



  RENT PAGE 1  →  https://www.propertyfinder.eg/ar/search?c=2&fu=0&rp=m&ob=mr&page=1
Found 25 cards
    1. فرصة مميزة للإيجار | توين هاوس فاخر في PK1
    2. شقة فاخرة للإيجار في كمبوند ووتر واي | موقع متميز
    3. دور أرضي مفروش مودرن للإيجار في ستون ريزيدانس
    4. هاسيندا وايت دوبلكس أرضى للإيجار 4 غرف وناني
    5. شقة للإيجار جنوب الأكاديمية أ – موقع ذهبي
    6. للإيجار – توين هاوس مميز صف أول جولف في ديونز
    7. شقة مفروشة بالكامل في فيليت سوديك
    8. فرش راقي | شارع ٩ | بجوار المترو
    9. فيلا للايجار بحمام سباحه كمبوند قطاميه ريزدنس
   10. توين هاوس للايجار 4غرف و غرفه في بينتهاوس باقل سعر
   11. اي فيلا رووف للإيجار بسعر مميز في ماونتن فيو
   12. شقة للإيجار في ليك فيو ريزيدنس | 144 متر
   13. شقة نصف مفروشة للإيجار في ميفيدا بلو فيوز | 185 م
   14. شقه للايجار بالمطبخ والتكييفات في النرجس 2 بالتجمع
   15. شقة للإيجار في سوديك إيستاون مساحة كبيرة
   16. إطلالة مفتوحة ورووف واسع لحياة هادئة ومميزة
   17. سنوديو مميز بالمطبخ والتكيفات للايجار في مراسم
   18. استديو مفروش فاخر 

,title,price,location,property_type,bedrooms,bathrooms,area,listing_type,agency_logo,agency_name,publish_date,publish_time,listing_url,image_urls,source
0,فرصة مميزة للإيجار | توين هاوس فاخر في PK1,٢٥٠٬٠٠٠ جنيه/ شهرياً,"بالم هيلز قطامية, كمبوندات القطامية, القطامية,...",منزل مزدوج,4,5,٤٠٠ متر مربع,Rent,https://static.shared.propertyfinder.eg/media/...,None,10.06.2026,11:12 PM,https://www.propertyfinder.eg/ar/plp/rent/twin...,https://static.shared.propertyfinder.eg/media/...,Property Finder Egypt
1,شقة فاخرة للإيجار في كمبوند ووتر واي | موقع متميز,١٠٬٠٠٠ جنيه/ شهرياً,"ووترواي إيست, التجمع الخامس, مدينة القاهرة الج...",شقة,3,4,٢٤٧ متر مربع,Rent,https://static.shared.propertyfinder.eg/media/...,None,11.06.2026,12:12 AM,https://www.propertyfinder.eg/ar/plp/rent/apar...,https://static.shared.propertyfinder.eg/media/...,Property Finder Egypt
2,دور أرضي مفروش مودرن للإيجار في ستون ريزيدانس,٧٢٬٠٠٠ جنيه/ شهرياً,"ستون ريزيدنس, كمبوندات التجمع الخامس, التجمع ا...",شقة,4,3,٢٠٠ متر مربع,Rent,https://static.shared.propertyfinder.eg/media/...,None,11.06.2026,12:12 AM,https://www.propertyfinder.eg/ar/plp/rent/apar...,https://static.shared.propertyfinder.eg/media/...,Property Finder Egypt
3,هاسيندا وايت دوبلكس أرضى للإيجار 4 غرف وناني,١٬١٠٠٬٠٠٠ جنيه/ شهرياً,"هاسييندا وايت, سيدي عبد الرحمن, الساحل الشمالي",دوبلكس,4,5,٢٠٠ متر مربع,Rent,https://static.shared.propertyfinder.eg/media/...,None,10.06.2026,10:12 PM,https://www.propertyfinder.eg/ar/plp/rent/dupl...,https://static.shared.propertyfinder.eg/media/...,Property Finder Egypt
4,شقة للإيجار جنوب الأكاديمية أ – موقع ذهبي,٥٠٬٠٠٠ جنيه/ شهرياً,"محور العروبة, المنطقة أ, حى جنوب الاكايمية, مد...",شقة,3,2,٢١٠ متر مربع,Rent,https://static.shared.propertyfinder.eg/media/...,None,10.06.2026,10:12 PM,https://www.propertyfinder.eg/ar/plp/rent/apar...,https://static.shared.propertyfinder.eg/media/...,Property Finder Egypt


## Cell 6 — Merge all results & save to CSV

Combines whichever DataFrames exist (`sale_df`, `rent_df`) into a
single file called **`propertyfinder_properties.csv`**.

> If you only ran one of the two scrapers, the missing DataFrame is simply skipped.


In [15]:
# ── Collect whichever DataFrames were created ────────────────────
COLUMNS = [
    "title", "price", "location", "property_type", "bedrooms", "bathrooms",
    "area", "listing_type", "agency_logo", "agency_name",
    "publish_date", "publish_time", "listing_url", "image_urls", "source",
]

available = {}
for name, var in [("Sale", "sale_df"), ("Rent", "rent_df")]:
    try:
        df_check = eval(var)
        if not df_check.empty:
            available[name] = df_check
            print(f"  ✅ {name:<6} — {len(df_check):>4} listings")
        else:
            print(f"  ⚠️  {name:<6} — DataFrame is empty, skipping")
    except NameError:
        print(f"  ℹ️  {name:<6} — not scraped yet, skipping")

if not available:
    print("\n❌ No data to merge. Run at least one scraper first.")
else:
    OUTPUT_FILE = "propertyfinder_properties.csv"

    combined = pd.concat(available.values(), ignore_index=True)
    existing_cols = [c for c in COLUMNS if c in combined.columns]
    combined = combined[existing_cols]

    combined.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print(f"\n{'─'*40}")
    print(f"  Total listings : {len(combined)}")
    print(f"  Saved to       : {OUTPUT_FILE}")
    print(f"{'─'*40}")

    combined.head(10)


  ✅ Sale   — 1250 listings
  ✅ Rent   — 1250 listings

────────────────────────────────────────
  Total listings : 2500
  Saved to       : propertyfinder_properties.csv
────────────────────────────────────────


In [16]:
import re

def normalize_logo_url(url):
    """شيل الـ query string (?v=...) والـ resolution (/178x98) من الـ URL."""
    if not url or str(url).startswith("data:"):
        return None
    url = str(url).split("?")[0]
    return re.sub(r'/\d+x\d+(\.\w+)$', r'\1', url)

# ── Normalise الاتنين ─────────────────────────────────────────────
combined["agency_logo_norm"]    = combined["agency_logo"].apply(normalize_logo_url)
agencies_df["agency_logo_norm"] = agencies_df["agency_logo"].apply(normalize_logo_url)

# ── Build lookup ──────────────────────────────────────────────────
logo_to_name = (
    agencies_df.dropna(subset=["agency_logo_norm"])
    .set_index("agency_logo_norm")["agency_name"]
    .to_dict()
)

# ── Update agency_name ────────────────────────────────────────────
combined["agency_name"] = combined["agency_logo_norm"].map(logo_to_name)
combined.drop(columns=["agency_logo_norm"], inplace=True)

# ── Save ──────────────────────────────────────────────────────────
combined.to_csv("propertyfinder_properties.csv", index=False, encoding="utf-8-sig")

# ── Summary ───────────────────────────────────────────────────────
matched   = combined["agency_name"].notna().sum()
no_logo   = combined["agency_logo"].isna().sum()
print(f"✅ Matched         : {matched}")
print(f"⚠️  No logo (NaN)  : {no_logo}")
combined[["agency_logo", "agency_name"]].dropna(subset=["agency_logo"]).head(10)

✅ Matched         : 2322
⚠️  No logo (NaN)  : 178


,agency_logo,agency_name
0,https://static.shared.propertyfinder.eg/media/...,Spade consultancy
1,https://static.shared.propertyfinder.eg/media/...,Property Hills One
2,https://static.shared.propertyfinder.eg/media/...,Spade consultancy
3,https://static.shared.propertyfinder.eg/media/...,Spade consultancy
4,https://static.shared.propertyfinder.eg/media/...,Spade consultancy
5,https://static.shared.propertyfinder.eg/media/...,Platinum Properties
6,https://static.shared.propertyfinder.eg/media/...,Platinum Properties
7,https://static.shared.propertyfinder.eg/media/...,Platinum Properties
8,https://static.shared.propertyfinder.eg/media/...,Platinum Properties
9,https://static.shared.propertyfinder.eg/media/...,NABAJ
